# Assignment: Bayesian Networks for Insurance Fraud Detection
### Groupe : Rémy Plastre, Pauline Rougeot, Romain Sebire 

Ce notebook présente l'implémentation et l'application d'un classifieur bayésien pour la détection de fraude dans les réclamations d'assurance automobile.

## Objectifs

1. Créer un classifieur scikit-learn personnalisé basé sur les réseaux bayésiens
2. Appliquer le classifieur au dataset d'assurance avec un pipeline scikit-learn
3. Comparer les performances avec un autre classifieur scikit-learn
4. Analyser les erreurs de classification via la structure du réseau bayésien appris

## Dataset

Le dataset utilisé provient de Kaggle et contient des données de réclamations d'assurance automobile avec des variables décrivant les caractéristiques des assurés, des véhicules et des sinistres.

## Configuration et imports

In [ ]:
# Imports 
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import multiprocessing as mp
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import validate_data, check_is_fitted
from sklearn.utils.multiclass import unique_labels
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.impute import SimpleImputer
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import HillClimbSearch, BayesianEstimator
from pgmpy.inference import VariableElimination
from tqdm import tqdm
import time
import logging
logging.getLogger('pgmpy').setLevel(logging.WARNING)

## 1. Implémentation du classifieur bayésien

Cette section présente l'implémentation d'un classifieur scikit-learn personnalisé basé sur les réseaux bayésiens. Le classifieur permet deux modes de fonctionnement :

- **Mode fourni** : Utilisation d'un modèle bayésien existant fourni par l'utilisateur
- **Mode automatique** : Apprentissage automatique de la structure du réseau via l'algorithme Hill Climbing

In [ ]:
# Question 1 : implémentation d'un classifieur Bayésien avec apprentissage de structure

class BayesianClassifier(ClassifierMixin, BaseEstimator):
    
    def __init__(self, model=None):
        self.model = model
        
    def fit(self, X, y):
        X, y = validate_data(self, X, y)
        self.classes_ = unique_labels(y)
        
        # Si un modèle est fourni par l'utilisateur, l'utiliser directement
        if self.model is not None:
            self.model.fit(X, y)
            self.learned_structure_ = False
            return self
        
        # Sinon, utiliser l'apprentissage de structure bayésienne par défaut
        # Convertir en DataFrame et discrétiser
        if not hasattr(X, 'columns'):
            X = pd.DataFrame(X, columns=[f'f_{i}' for i in range(X.shape[1])])
        
        # Features
        X_full = X.copy()
        
        # Cible
        data = pd.concat([X_full, pd.Series(y, name='target')], axis=1)
        
        # Hill Climbing 
        hc = HillClimbSearch(data)
        model_structure = hc.estimate(
            max_iter=1000,        # Plus d'itérations
            max_indegree=3,       # Commencer plus petit
            tabu_length=5,        # Éviter les cycles
            epsilon=1e-4,          # Permet jusqu'à 10 parents par nœud (au lieu de 3 par défaut)
            show_progress=True,     # Afficher la barre de progression
        )
        
        # Créer le réseau bayésien
        self.model = DiscreteBayesianNetwork(model_structure.edges())
        estimator = BayesianEstimator(self.model, data)
        for node in self.model.nodes():
            self.model.add_cpds(estimator.estimate_cpd(node))
        
        self.inference_ = VariableElimination(self.model)
        self.learned_structure_ = True
        self.feature_names_ = X_full.columns.tolist()
        
        return self
    
    def predict(self, X):
        check_is_fitted(self)
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]
    
    def predict_proba(self, X):
        check_is_fitted(self)
        
        # Si modèle fourni par l'utilisateur
        if not hasattr(self, 'learned_structure_') or not self.learned_structure_:
            return self.model.predict_proba(X)
        
        # Si apprentissage de structure bayésienne
        X_full = X.copy() if hasattr(X, 'columns') else pd.DataFrame(X, columns=self.feature_names_)
        if not hasattr(X_full, 'columns'):
            X_full = pd.DataFrame(X_full, columns=self.feature_names_)
        
        probabilities = []
        for i in range(len(X_full)):
            valid_evidence = {}
            
            for col in X_full.columns:
                if col in self.model.nodes() and col != 'target':
                    raw_value = X_full.iloc[i][col]
                    
                    try:
                        # Utiliser directement la valeur numérique (float ou int selon le cas)
                        if isinstance(raw_value, (int, float)):
                            # Préférer int si c'est un entier, sinon float
                            if float(raw_value).is_integer():
                                value_format = int(raw_value)
                            else:
                                value_format = float(raw_value)
                        else:
                            # Si ce n'est pas numérique, essayer de le convertir
                            numeric_val = float(raw_value)
                            if numeric_val.is_integer():
                                value_format = int(numeric_val)
                            else:
                                value_format = numeric_val
                        
                        # Utiliser directement la valeur convertie
                        valid_evidence[col] = value_format
                        
                    except (ValueError, TypeError):
                        # Si la conversion échoue, ignorer cette variable
                        continue
            
            # Faire la prédiction avec l'evidence valide
            try:
                if valid_evidence:
                    result = self.inference_.query(['target'], evidence=valid_evidence)
                    probabilities.append(result.values)
                else:
                    # Si aucune evidence valide, utiliser les probabilités marginales
                    marginal = self.inference_.query(['target'])
                    probabilities.append(marginal.values)
            except:
                # En dernier recours, probabilité uniforme
                uniform_prob = np.ones(len(self.classes_)) / len(self.classes_)
                probabilities.append(uniform_prob)
    
        return np.array(probabilities)

## 2. Application du classifieur au dataset d'assurance

Cette section applique le classifieur bayésien au dataset de réclamations d'assurance automobile en utilisant un pipeline scikit-learn complet. Le pipeline inclut :

- **Préprocessing des données** : Gestion des valeurs manquantes, discrétisation des variables numériques, encodage des variables catégorielles
- **Entraînement** : Application du classifieur bayésien avec apprentissage automatique de structure
- **Évaluation** : Mesure des performances sur un ensemble de test

In [ ]:
# Question 2 : application du classifieur Bayésien au dataset 

# Wrapper pour LabelEncoder compatible avec Pipeline et valeurs inconnues
class MultiLabelEncoder(BaseEstimator, ClassifierMixin):
    def __init__(self):
        self.label_encoders = {}
    
    def fit(self, X, y=None):
        # Convertir en DataFrame si nécessaire
        if not hasattr(X, 'columns'):
            X = pd.DataFrame(X)
        
        for i, col in enumerate(X.columns):
            le = LabelEncoder()
            # Ajouter une catégorie spéciale pour les valeurs inconnues
            unique_values = list(X.iloc[:, i].astype(str).unique()) + ['__UNKNOWN__']
            le.fit(unique_values)
            self.label_encoders[i] = le
        return self
    
    def transform(self, X):
        # Convertir en DataFrame si nécessaire
        if not hasattr(X, 'columns'):
            X = pd.DataFrame(X)
        
        X_encoded = X.copy()
        for i, col in enumerate(X.columns):
            if i in self.label_encoders:
                # Remplacer les valeurs inconnues par '__UNKNOWN__'
                values = X.iloc[:, i].astype(str)
                known_values = set(self.label_encoders[i].classes_)
                values_safe = [v if v in known_values else '__UNKNOWN__' for v in values]
                X_encoded.iloc[:, i] = self.label_encoders[i].transform(values_safe)
        
        # Retourner en array numpy pour compatibilité pipeline
        return X_encoded.values
    
    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

# Charger le dataset
df = pd.read_csv('insurance_claims.csv')

# Définir la cible
target = 'fraud_reported' if 'fraud_reported' in df.columns else df.columns[-1]
X = df.drop(columns=[target])
y = df[target]

# Identifier les colonnes numériques et catégorielles
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# Pipeline scikit-learn avec LabelEncoder (évite l'explosion combinatoire)
pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('discretizer', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile'))
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', MultiLabelEncoder())
        ]), cat_cols)
    ])),
    ('classifier', BayesianClassifier())
])

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entraînement 
print(" Démarrage de l'entraînement du BayesianClassifier...")

# Barre de progression pour l'entraînement
with tqdm(total=100, desc="Entraînement", bar_format='{l_bar}{bar}| {percentage:3.0f}%') as pbar:
    pbar.set_description("📚 Préparation des données")
    pbar.update(10)
    time.sleep(0.1)
    
    pbar.set_description("🔍 Apprentissage de structure")
    pbar.update(20)
    
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    pbar.set_description("✅ Entraînement terminé")
    pbar.update(70)

print(f"⏱️  Temps d'entraînement: {train_time:.2f}s")

# Barre de progression pour la prédiction
print("\n Prédictions en cours...")
with tqdm(total=100, desc="Prédiction", bar_format='{l_bar}{bar}| {percentage:3.0f}%') as pbar:
    start_pred = time.time()
    y_pred = pipeline.predict(X_test)
    pred_time = time.time() - start_pred
    pbar.update(100)

print(f"⏱️  Temps de prédiction: {pred_time:.2f}s")

# Évaluation
print("\n Résultats:")
accuracy = accuracy_score(y_test, y_pred)
print(f" Accuracy: {accuracy:.4f}")
print("\n Rapport de classification:")
print(classification_report(y_test, y_pred))

 Démarrage de l'entraînement du BayesianClassifier...


🔍 Apprentissage de structure:  30%|███       |  30%

  0%|          | 0/1000 [00:00<?, ?it/s]

✅ Entraînement terminé: 100%|██████████| 100%      


⏱️  Temps d'entraînement: 1.28s

 Prédictions en cours...


Prédiction: 100%|██████████| 100%

⏱️  Temps de prédiction: 0.15s

 Résultats:
 Accuracy: 0.8000

 Rapport de classification:
              precision    recall  f1-score   support

           N       0.84      0.90      0.87       151
           Y       0.62      0.49      0.55        49

    accuracy                           0.80       200
   macro avg       0.73      0.70      0.71       200
weighted avg       0.79      0.80      0.79       200



## 3. Comparaison avec RandomForest

Cette section compare les performances du classifieur bayésien avec un autre classifieur scikit-learn standard : RandomForest. La comparaison se fait sur les mêmes données préprocessées pour assurer une évaluation équitable.

In [ ]:
# Question 3 : comparaison avec RandomForest + validation croisée stratifiée

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Pipeline RandomForest avec parallélisation
rf_pipeline = Pipeline([
    ('preprocessor', pipeline.named_steps['preprocessor']),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Entraînement et test simple d'abord
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("=== COMPARAISON TRAIN/TEST ===")
print(f"BayesianClassifier: {accuracy_score(y_test, y_pred):.4f}")
print(f"RandomForest:       {accuracy_score(y_test, y_pred_rf):.4f}")

=== COMPARAISON TRAIN/TEST ===
BayesianClassifier: 0.7550
RandomForest:       0.7850


## 4. Analyse des erreurs de classification

Cette section analyse les erreurs de classification en utilisant la structure du réseau bayésien appris. L'objectif est de comprendre pourquoi certaines instances sont mal classifiées en examinant :

- La structure du réseau bayésien (noeuds et arêtes)
- Les probabilités conditionnelles pour les instances mal classifiées
- L'influence des variables parents sur la variable cible

In [ ]:
# Question 4 : Interprétation des erreurs de classification via la structure du réseau bayésien

# Obtenir les prédictions et erreurs
y_pred_final = pipeline.predict(X_test)
errors_mask = y_test != y_pred_final
error_indices = y_test[errors_mask].index.tolist()

print(f"Total erreurs: {errors_mask.sum()}/{len(y_test)} ({errors_mask.sum()/len(y_test)*100:.1f}%)")

# Sélectionner 5 erreurs
step = len(error_indices) // 5 if len(error_indices) >= 5 else 1
selected_errors = [error_indices[i*step] for i in range(min(5, len(error_indices)))]

# Obtenir le classifieur et sa structure
clf = pipeline.named_steps['classifier']
target_parents = clf.model.get_parents('target')
print(f"Structure réseau: {len(clf.model.nodes())} noeuds, parents de target: {target_parents}")

# ANALYSE DE LA STRUCTURE BAYÉSIENNE
print(f"\nAnalyse structurelle:")
print(f"Variable principale influençant target: {target_parents[0] if target_parents else 'Aucune'}")

# Probabilité marginale de target
marginal = clf.inference_.query(['target'])
prob_marginal = dict(zip(clf.classes_, marginal.values))
print(f"Probabilité marginale P(target): {prob_marginal}")

# Probabilité conditionnelle du parent principal
if target_parents:
    parent_main = target_parents[0]
    prob_parent_0 = clf.inference_.query(['target'], evidence={parent_main: 0.0})
    prob_parent_1 = clf.inference_.query(['target'], evidence={parent_main: 1.0})
    print(f"P(target|{parent_main}=0): {dict(zip(clf.classes_, prob_parent_0.values))}")
    print(f"P(target|{parent_main}=1): {dict(zip(clf.classes_, prob_parent_1.values))}")

# Analyser directement les 5 erreurs
print("\nAnalyse des erreurs:")

# Erreur 1
idx = selected_errors[0]
true_val = y_test.loc[idx]
pred_val = y_pred_final[y_test.index.get_loc(idx)]
X_sample = pipeline.named_steps['preprocessor'].transform(X_test.loc[[idx]])[0]

X_df = pd.DataFrame([X_sample], columns=clf.feature_names_)
evidence = {}
for col in X_df.columns:
    if col in clf.model.nodes() and col != 'target':
        val = X_df.iloc[0][col]
        evidence[col] = int(val) if float(val).is_integer() else float(val)

conditional = clf.inference_.query(['target'], evidence=evidence)
prob_cond = dict(zip(clf.classes_, conditional.values))
print(f"Erreur 1: Prédit={pred_val}, Vrai={true_val}, P(N|données)={prob_cond['N']:.3f}, P(Y|données)={prob_cond['Y']:.3f}")
if target_parents:
    print(f"  -> {target_parents[0]}={evidence.get(target_parents[0], 'N/A')}")

# Erreur 2
idx = selected_errors[1]
true_val = y_test.loc[idx]
pred_val = y_pred_final[y_test.index.get_loc(idx)]
X_sample = pipeline.named_steps['preprocessor'].transform(X_test.loc[[idx]])[0]

X_df = pd.DataFrame([X_sample], columns=clf.feature_names_)
evidence = {}
for col in X_df.columns:
    if col in clf.model.nodes() and col != 'target':
        val = X_df.iloc[0][col]
        evidence[col] = int(val) if float(val).is_integer() else float(val)

conditional = clf.inference_.query(['target'], evidence=evidence)
prob_cond = dict(zip(clf.classes_, conditional.values))
print(f"Erreur 2: Prédit={pred_val}, Vrai={true_val}, P(N|données)={prob_cond['N']:.3f}, P(Y|données)={prob_cond['Y']:.3f}")
if target_parents:
    print(f"  -> {target_parents[0]}={evidence.get(target_parents[0], 'N/A')}")

# Erreur 3
idx = selected_errors[2]
true_val = y_test.loc[idx]
pred_val = y_pred_final[y_test.index.get_loc(idx)]
X_sample = pipeline.named_steps['preprocessor'].transform(X_test.loc[[idx]])[0]

X_df = pd.DataFrame([X_sample], columns=clf.feature_names_)
evidence = {}
for col in X_df.columns:
    if col in clf.model.nodes() and col != 'target':
        val = X_df.iloc[0][col]
        evidence[col] = int(val) if float(val).is_integer() else float(val)

conditional = clf.inference_.query(['target'], evidence=evidence)
prob_cond = dict(zip(clf.classes_, conditional.values))
print(f"Erreur 3: Prédit={pred_val}, Vrai={true_val}, P(N|données)={prob_cond['N']:.3f}, P(Y|données)={prob_cond['Y']:.3f}")
if target_parents:
    print(f"  -> {target_parents[0]}={evidence.get(target_parents[0], 'N/A')}")

# Erreur 4
idx = selected_errors[3]
true_val = y_test.loc[idx]
pred_val = y_pred_final[y_test.index.get_loc(idx)]
X_sample = pipeline.named_steps['preprocessor'].transform(X_test.loc[[idx]])[0]

X_df = pd.DataFrame([X_sample], columns=clf.feature_names_)
evidence = {}
for col in X_df.columns:
    if col in clf.model.nodes() and col != 'target':
        val = X_df.iloc[0][col]
        evidence[col] = int(val) if float(val).is_integer() else float(val)

conditional = clf.inference_.query(['target'], evidence=evidence)
prob_cond = dict(zip(clf.classes_, conditional.values))
print(f"Erreur 4: Prédit={pred_val}, Vrai={true_val}, P(N|données)={prob_cond['N']:.3f}, P(Y|données)={prob_cond['Y']:.3f}")
if target_parents:
    print(f"  -> {target_parents[0]}={evidence.get(target_parents[0], 'N/A')}")

# Erreur 5
idx = selected_errors[4]
true_val = y_test.loc[idx]
pred_val = y_pred_final[y_test.index.get_loc(idx)]
X_sample = pipeline.named_steps['preprocessor'].transform(X_test.loc[[idx]])[0]

X_df = pd.DataFrame([X_sample], columns=clf.feature_names_)
evidence = {}
for col in X_df.columns:
    if col in clf.model.nodes() and col != 'target':
        val = X_df.iloc[0][col]
        evidence[col] = int(val) if float(val).is_integer() else float(val)

conditional = clf.inference_.query(['target'], evidence=evidence)
prob_cond = dict(zip(clf.classes_, conditional.values))
print(f"Erreur 5: Prédit={pred_val}, Vrai={true_val}, P(N|données)={prob_cond['N']:.3f}, P(Y|données)={prob_cond['Y']:.3f}")
if target_parents:
    print(f"  -> {target_parents[0]}={evidence.get(target_parents[0], 'N/A')}")

# INTERPRÉTATION BASÉE SUR LA STRUCTURE
print(f"\nInterprétation via structure bayésienne:")
print(f"Le réseau utilise principalement {target_parents[0] if target_parents else 'aucune variable'} pour prédire target")
print(f"Structure simple: {len(target_parents)} parent(s) direct(s) de target sur {len(clf.model.nodes())} variables")

Total erreurs: 49/200 (24.5%)
Structure réseau: 16 noeuds, parents de target: ['f_29']

Analyse structurelle:
Variable principale influençant target: f_29
Probabilité marginale P(target): {'N': 0.7509316770186334, 'Y': 0.2490683229813665}
P(target|f_29=0): {'N': 0.40238365493757094, 'Y': 0.5976163450624291}
P(target|f_29=1): {'N': 0.8985828166519043, 'Y': 0.10141718334809566}

Analyse des erreurs:
Erreur 1: Prédit=N, Vrai=Y, P(N|données)=0.437, P(Y|données)=0.563
  -> f_29=0
Erreur 2: Prédit=N, Vrai=Y, P(N|données)=0.313, P(Y|données)=0.687
  -> f_29=1
Erreur 3: Prédit=N, Vrai=Y, P(N|données)=0.453, P(Y|données)=0.547
  -> f_29=0
Erreur 4: Prédit=N, Vrai=Y, P(N|données)=0.568, P(Y|données)=0.432
  -> f_29=0
Erreur 5: Prédit=N, Vrai=Y, P(N|données)=0.313, P(Y|données)=0.687
  -> f_29=1

Interprétation via structure bayésienne:
Le réseau utilise principalement f_29 pour prédire target
Structure simple: 1 parent(s) direct(s) de target sur 16 variables


## Conclusion

Dans ce notebook, nous avons implémenté un classifieur bayésien pour la détection de fraude dans les réclamations d'assurance, ce qui nous a permis de réaliser les points suivants :

1. **Classifieur personnalisé** : Implémentation conforme au standard scikit-learn avec apprentissage automatique de structure
2. **Pipeline complet** : Préprocessing adapté aux données mixtes (numériques et catégorielles)
3. **Comparaison** : Benchmark avec RandomForest pour évaluer les performances
4. **Interprétabilité** : Analyse des erreurs via la structure bayésienne apprise 

Ces résultats démontrent l'efficacité et l'interprétabilité des réseaux bayésiens pour la détection de fraude dans les réclamations d'assurance automobile.